## Paraná Basin – Real Data

In this notebook, we apply the workflow to real magnetic data from the Paraná Basin.  
The dataset, provided by ANP, is available through the [CPRM/REATE portal](https://reate.cprm.gov.br/anp/TERRESTRE).

The data can be automatically downloaded within this notebook, where it is also preprocessed.  
As part of the preparation, the sampling rate is reduced from 100 Hz to 1 Hz  
(see `02-parana-basin-data-preparation.ipynb`).


In [ ]:
import os
import pygmt
import pooch
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import bordado as bd
import boule as bl
import harmonica as hm
import verde as vd
import itertools
from scipy.interpolate import griddata

import spherical as sph

Import the real data.


In [ ]:
caminho = "../data/parana-basin-magnetic-processed.csv"

df = pd.read_csv(caminho, sep=',', comment="#")
df.columns = ['FID','ESTACAO','LINHA','X','Y','LATITUDE','LONGITUDE','DATA','HORA','GPSALT','BARO','MAGBRU','MAGCOM','MAGBASE','MAGCOR','MAGNIV','MAGMIC','MAGIGRF','IGRF','MDT']

Plot the real data.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-57, -46, -26, -13],
    projection="M20c", 
    frame="afg",
    land="gray"
)

scale = vd.maxabs(df.MAGIGRF)
pygmt.makecpt(cmap="polar+h", series=[-scale, scale])

fig.plot(
    x=df.LONGITUDE,
    y=df.LATITUDE,
    fill=df.MAGIGRF,
    style="c0.05c",
    cmap=True
)

fig.colorbar(position="JBC")
fig.show(width=500)

The dataset contains inconsistencies along the tie lines.  
These tie flight lines are identified and removed to ensure data quality.

In [ ]:
prefixos = ('19', '29', '39')
df = df[~df['LINHA'].astype(str).str.startswith(prefixos)].copy()

fig = pygmt.Figure()

fig.coast(
    region=[-57, -46, -26, -13],
    projection="M20c", 
    frame="afg",
    land="gray"
)

scale = 210
pygmt.makecpt(cmap="polar+h", series=[-scale, scale])

fig.plot(
    x=df.LONGITUDE,
    y=df.LATITUDE,
    fill=df.MAGIGRF,
    style="c0.05c",
    cmap=True
)

fig.colorbar(position="JBC")
fig.show(width=500)

As in the previous notebook with the synthetic data, we reduce the dataset to speed up computations and improve processing efficiency.

In [ ]:
reducer = vd.BlockReduce(reduction=np.median, spacing= 0.0023)

coordinates, mag= reducer.filter(
    (df.LONGITUDE, df.LATITUDE, df.GPSALT), df.MAGIGRF
)

coordinates, height = reducer.filter(
    (df.LONGITUDE, df.LATITUDE), df.GPSALT
)

lon, lat = coordinates

coordinates = (lon, lat, height)

# coordinates_real = (df['LONGITUDE'].values, df['LATITUDE'].values, df["GPSALT"].values)
# mag_real = (df['MAGIGRF'].values)

print(coordinates)
# print(coordinates_real[0].size)

In [ ]:
region_croped = (-54, -50, -25, -16)

crop = bd.inside((coordinates[0], coordinates[1]), region_croped)

lon_crop = coordinates[0][crop]
lat_crop= coordinates[1][crop]
height_crop = coordinates[2][crop]
mag_crop = mag[crop]

coordinates = (lon_crop, lat_crop, height_crop)

# Deep equivalent sources
Use a block reduce method to create our deep equivalent sources.

In [ ]:
reducer = vd.BlockReduce(reduction=np.mean, spacing=0.1,  center_coordinates=False, drop_coords=False)
blocked_deep_equivalent_sources, magnetic_anomaly_reduced = reducer.filter(coordinates, mag_crop)

print(blocked_deep_equivalent_sources[2].size)

Plot the data so we can visualize the reduced version.

In [ ]:
plt.scatter(*blocked_deep_equivalent_sources[:2], c = magnetic_anomaly_reduced,s = 5, cmap="seismic", vmin=-210, vmax=210)
plt.colorbar()

In [ ]:
def estimate_depth(coordinates):
        """
        Estimate a reasonable depth if one isn't given.
        """
        ellipsoid=bl.WGS84

        coslat = np.cos(np.radians(coordinates[1]))
        sinlat = np.sin(np.radians(coordinates[1]))
        N = ellipsoid.prime_vertical_radius(sinlat)
        b = ellipsoid.semiminor_axis
        a = ellipsoid.semimajor_axis
        coordinates_cartesian = (
            (N + coordinates[2]) * coslat * np.cos(np.radians(coordinates[0])),
            (N + coordinates[2]) * coslat * np.sin(np.radians(coordinates[0])),
            (b**2 * N / a**2 + coordinates[2]) * sinlat,
        )
        return bd.neighbor_distance_statistics(coordinates_cartesian, "median", k=10)

Perform cross-validation to select the best equivalent source model for the dataset.

In [ ]:
inclination, declination = -25, -20 
damping_deep = [1e1, 1e2, 1e3, 1e4]
depths = np.mean(estimate_depth(blocked_deep_equivalent_sources))
source_depth_deep = [180e3, 200e3, 220e3, 240e3] #depths*2.5, depths*4.25, depths*5.75
parameter_sets_deep = [
    dict(damping=combo[0], depth=combo[1])
    for combo in itertools.product(damping_deep, source_depth_deep)
]
print("Number of combinations:", len(parameter_sets_deep))

In [ ]:
%%time
kfold = vd.BlockKFold(
    spacing=1.5,
    shuffle=True,
    random_state=0,
    balance=True,
)
features = np.transpose(blocked_deep_equivalent_sources[:2])
scores_deep = []
damping_deep = []
source_depth_deep = []
for parameters in parameter_sets_deep:    
    print(parameters)
    eqs_deep = sph.EquivalentSourcesMagGeod(**parameters)
    tmp = []
    for train, test in kfold.split(features):
        eqs_deep.fit(
            [c[train] for c in blocked_deep_equivalent_sources],
            inclination, 
            declination,
            magnetic_anomaly_reduced[train] 
        )
        predicted = hm.total_field_anomaly(
            eqs_deep.predict([c[test] for c in blocked_deep_equivalent_sources]),
            inclination, declination
        )
        tmp.append(np.linalg.norm(magnetic_anomaly_reduced[test] - predicted))
    scores_deep.append(np.mean(tmp))
    damping_deep.append(parameters['damping'])
    source_depth_deep.append(parameters['depth'])
best = np.argmin(scores_deep)
parameter_sets_deep[best]

In [ ]:
damping_values_deep = np.array(damping_deep)
depth_values_deep = np.array(source_depth_deep)
score_values_deep = np.array(scores_deep)

best_deep = np.argmin(score_values_deep)
best_damping_deep = damping_values_deep[best_deep]
best_depth_deep = depth_values_deep[best_deep]
best_rmse_deep = score_values_deep[best_deep]
best_params_deep = parameter_sets_deep[best_deep]
print("Best parameters:", best_params_deep)
print(f"Best RMSE: {best_rmse_deep:.3f} nT")

x_deep = np.logspace(np.log10(damping_values_deep.min()), np.log10(damping_values_deep.max()), 200)
y_deep = np.linspace(depth_values_deep.min(), depth_values_deep.max(), 200)
X_deep, Y_deep = np.meshgrid(x_deep, y_deep)
Z_deep = griddata(
    (damping_values_deep, depth_values_deep),
    score_values_deep,
    (X_deep, Y_deep),
    method='linear',
)

plt.figure(figsize=(8, 6))
levels_deep = np.linspace(
    Z_deep.min(),
    np.nanpercentile(score_values_deep, 80),
    20,
)
c = plt.contourf(
    X_deep,
    Y_deep,
    Z_deep,
    levels=levels_deep,
    cmap="viridis",
    extend='max',
)
plt.plot(
    best_damping_deep,
    best_depth_deep,
    '*',
    color='white',
    markersize=15,
    label='Lowest RMSE'
)
plt.xscale('log')
plt.xlabel('Damping')
plt.ylabel('Source Relative Depth (m)')
plt.colorbar(c, label='RMSE (nT)')
plt.legend()
plt.title("Deep Layer")
plt.show()

Run the inversion of the deep equivalent sources with the values obtained with the cross-validation.

In [ ]:
%%time
eqs_deep = sph.EquivalentSourcesMagGeod(**parameter_sets_deep[best])
eqs_deep.fit(blocked_deep_equivalent_sources, inclination, declination, magnetic_anomaly_reduced)


In [ ]:
predicted_deep =  hm.total_field_anomaly(eqs_deep.predict(coordinates), inclination, declination)
residuals_deep = mag_crop - predicted_deep 
predicted_blocked_deep = eqs_deep.predict(blocked_deep_equivalent_sources)

# residuals_be_deep = magnetic_field[0].ravel() - predicted_b_deep[0]
# residuals_bn_deep = magnetic_field[1].ravel() - predicted_b_deep[1]
# residuals_bu_deep = magnetic_field[2].ravel() - predicted_b_deep[2]

print(residuals_deep.size)

Plot the predidted and the residuals of the deep layer, to check if the EQS is good.

In [ ]:
scale_deep = vd.maxabs(predicted_deep)
plt.figure(figsize=(15,3))


plt.subplot(1,2,1)
plt.scatter(coordinates[0], coordinates[1],c = predicted_deep, s = 5,cmap="seismic", vmin=-scale_deep, vmax=scale_deep)
plt.colorbar(label='nT')

plt.subplot(1,2,2)
plt.scatter(coordinates[0], coordinates[1],c = residuals_deep, s = 5,cmap="seismic", vmin=-scale_deep, vmax=scale_deep)
plt.colorbar(label='nT')

# Shallow equivalent sources
Use the gradient-boosted method to generate our shallow equivalent sources.

But frist we will crop a sub-area of our data to make cross-validation run faster as done in Soler & Uieda (2021).

In [ ]:
#For the reduced data
region_croped = (-54, -50, -18, -16)

cv = bd.inside((coordinates[0], coordinates[1]), region_croped)

lon_cv = coordinates[0][cv]
lat_cv = coordinates[1][cv]
height_cv = coordinates[2][cv]
mag_cv = mag_crop[cv]
residuals_deep_cv = residuals_deep[cv]

coordinates_cv = (lon_cv, lat_cv, height_cv)

# #For the real data

# region_croped = (-52, -50, -18, -16)

# cv = bd.inside((coordinates[0], coordinates_real[1]), region_croped)

# lon_cv = coordinates_real[0][cv]
# lat_cv = coordinates_real[1][cv]
# height_cv = coordinates_real[2][cv]
# mag_cv = mag_real[cv]
# residuals_deep_cv = residuals_deep[cv]

# coordinates_cv = (lon_cv, lat_cv, height_cv)

# print(coordinates_cv[0].size)
# print(residuals_deep_cv.size)

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-54.5, -49.5, -18.5, -15.5],
    projection="M20c", 
    frame="afg",
    land="gray"
)

scale = 210
pygmt.makecpt(cmap="polar+h", series=[-scale, scale])

fig.plot(
    x=coordinates_cv[0],
    y=coordinates_cv[1],
    fill=mag_cv,
    style="c0.1c",
    cmap=True
)

fig.colorbar(position="JBC")
fig.show(width=700)

In [ ]:
dampings_shallow = [1e-5,1e-2,1e0,1e2, 1e5]
depths_shallow = [27e3, 29e3, 31e3, 40e3]
parameter_sets_shallow = [
    dict(damping=combo[0], depth=combo[1])
    for combo in itertools.product(dampings_shallow, depths_shallow)
] 
print("Number of combinations:", len(parameter_sets_shallow))

In [ ]:
%%time
kfold = vd.BlockKFold(
    spacing=0.5,
    shuffle=True,
    random_state=0,
    balance=True,
)
features = np.transpose(coordinates_cv[:2])
scores_shallow = []
damping_shallow = []
source_depth_shallow = []
for parameters in parameter_sets_shallow:    
    print(parameters)
    eqs_shallow = sph.EquivalentSourcesMagGeodGB(window_size=1,verbose=False,**parameters)
    tmp = []
    for train, test in kfold.split(features):
        eqs_shallow.fit(
            [c[train] for c in coordinates_cv],  
            inclination,declination,
            residuals_deep_cv[train],
        )
        predicted = hm.total_field_anomaly(
            eqs_shallow.predict([c[test] for c in coordinates_cv]),
            inclination,declination,
        )
        tmp.append(np.linalg.norm(residuals_deep_cv[test] - predicted))
    scores_shallow.append(np.mean(tmp))
    damping_shallow.append(parameters['damping'])
    source_depth_shallow.append(parameters['depth'])
best = np.argmin(scores_shallow)
parameter_sets_shallow[best]

In [ ]:
damping_values_shallow = np.array(damping_shallow)
depth_values_shallow = np.array(source_depth_shallow)
score_values_shallow = np.array(scores_shallow)

best_shallow = np.argmin(score_values_shallow)
best_damping_shallow = damping_values_shallow[best_shallow]
best_depth_shallow = depth_values_shallow[best_shallow]
best_rmse_shallow = score_values_shallow[best_shallow]
best_params_shallow = parameter_sets_shallow[best_shallow]
print("Best parameters:", best_params_shallow)
print(f"Best RMSE: {best_rmse_shallow:.3f} nT")

x_shallow = np.logspace(np.log10(damping_values_shallow.min()), np.log10(damping_values_shallow.max()), 100)
y_shallow = np.linspace(depth_values_shallow.min(), depth_values_shallow.max(), 100)
X_shallow, Y_shallow = np.meshgrid(x_shallow, y_shallow)

Z_shallow = griddata(
    (damping_values_shallow, depth_values_shallow),
    score_values_shallow,
    (X_shallow, Y_shallow),
    method='linear'
)
plt.figure(figsize=(8, 6))
levels_shallow = np.linspace(
    Z_shallow.min(),
    np.nanpercentile(score_values_shallow, 80),
    20,
)
c = plt.contourf(
    X_shallow,
    Y_shallow,
    Z_shallow,
    levels=levels_shallow,
    cmap="viridis",
    extend='max',
)
plt.plot(
    best_damping_shallow,
    best_depth_shallow,
    '*',
    color='white',
    markersize=15,
    label='Lowest RMSE'
)
plt.xscale('log')
plt.xlabel('Damping')
plt.ylabel('Source Relative Depth (m)')
plt.colorbar(c, label='RMSE (nT)')
plt.legend()
plt.title("Shallow Layer")
plt.show()

In [ ]:
%%time
eqs_shallow = sph.EquivalentSourcesMagGeodGB(**parameter_sets_shallow[best])
eqs_shallow.fit(coordinates, inclination, declination, residuals_deep)
print(eqs_shallow.window_size_)

In [ ]:
predicted_shallow =  hm.total_field_anomaly(eqs_shallow.predict(coordinates), inclination, declination)
residuals_shallow = residuals_deep - predicted_shallow 

predicted_b_shallow = eqs_shallow.predict(coordinates)

In [ ]:
scale_deep = vd.maxabs(predicted_shallow)
plt.figure(figsize=(15,3))


plt.subplot(1,2,1)
plt.scatter(coordinates[0], coordinates[1],c = predicted_shallow, s = 5,cmap="seismic", vmin=-scale, vmax=scale)
plt.colorbar(label='nT')

plt.subplot(1,2,2)
plt.scatter(coordinates[0], coordinates[1],c = residuals_shallow, s = 5,cmap="seismic", vmin=-scale, vmax=scale)
plt.colorbar(label='nT')

In [ ]:
predicted_tfa = predicted_deep + predicted_shallow
residuals_tfa = predicted_tfa - mag_crop

scale_deep = vd.maxabs(predicted_tfa)
plt.figure(figsize=(15,3))


plt.subplot(1,2,1)
plt.scatter(coordinates[0], coordinates[1],c = predicted_tfa, s = 5,cmap="seismic", vmin=-scale, vmax=scale)
plt.colorbar(label='nT')

plt.subplot(1,2,2)
plt.scatter(coordinates[0], coordinates[1],c = residuals_shallow, s = 5,cmap="seismic", vmin=-scale, vmax=scale)
plt.colorbar(label='nT')

In [ ]:
shapefile_brasil = "../data/BR_UF_2022/BR_UF_2022.shp"


required_extensions = ['.shp', '.shx', '.dbf']
base_name = os.path.splitext(shapefile_brasil)[0]

missing_files = []
for ext in required_extensions:
    file_path = f"{base_name}{ext}"
    if not os.path.exists(file_path):
        missing_files.append(file_path)

if missing_files:
    print(f"Erro: Os seguintes arquivos obrigatórios estão faltando: {', '.join(missing_files)}")
    exit()


try:
    brasil = gpd.read_file(shapefile_brasil)
except Exception as e:
    print(f"Erro ao carregar o shapefile: {e}")
    exit()


brasil_reduc = brasil.simplify(0.05)

In [ ]:
residuals_tfa = mag_real - (predicted_deep + predicted_shallow)
shallow = mag_real - predicted_deep
residuals_blocked_deep = predicted_blocked_deep - magnetic_anomaly_reduced

plot_data = [
    (mag_real, coordinates[0], coordinates[1], "Observed Total-field anomaly"),
    (predicted_blocked_deep[0], blocked_deep_equivalent_sources[0], blocked_deep_equivalent_sources[1], "Deep Total-field anomaly"),
    (shallow, coordinates[0], coordinates[1], "Shallow Total-field anomaly"),
    (residuals_tfa, coordinates[0], coordinates[1], "Residuals"),
    (residuals_blocked_deep[0], blocked_deep_equivalent_sources[0], blocked_deep_equivalent_sources[1], "Residuals Deep"),
    (residuals_shallow, coordinates[0], coordinates[1], "Residuals Shallow")
]

region = [-57, -46.5, -25, -13.5]
scale = 110
fig = pygmt.Figure()

with pygmt.config(FONT="10p,Palatino-Roman",
    FONT_ANNOT="8p,Palatino-Roman",
    FONT_ANNOT_PRIMARY="8p,Palatino-Roman",
    FONT_ANNOT_SECONDARY="8p,Palatino-Roman",
    MAP_FRAME_WIDTH="1.5p",
    MAP_TITLE_OFFSET="-2p", ):

    pygmt.makecpt(cmap="polar", series=[-scale, scale])

    with fig.subplot(nrows=2,
            ncols=3,
            figsize=("16c", "12.5c"),
            autolabel="+jTL+o0.1/0.3c",
            margins='0.2c/.2c',
            sharex="b",
            sharey="l"):

        for i, (data, x_coords, y_coords, title) in enumerate(plot_data):
            with fig.set_panel(panel=i):

                fig.coast(
                    land="#dddddd",
                    region = region,
                    projection="M?",
                )

                fig.plot(
                    x=x_coords,
                    y=y_coords,
                    fill=data,
                    style="c0.03c",
                    projection="M?",
                    cmap=True
                )

                fig.plot(
                    data=brasil_reduc.geometry,
                    pen=".2p,white",
                    fill=None,
                    projection="M?",
                    frame=["a42f"]
                )

                fig.basemap(frame=f"+t{title}")

pygmt.config(FONT_ANNOT_PRIMARY="10p,Palatino-Roman", FONT_ANNOT_SECONDARY="10p,Palatino-Roman")
fig.colorbar(position="JBC+w10/0.3c+h", frame=["y+lnT"])
fig.show(width=1000)